# 03. Text OCR (Model A - Step 2)

`02_layout_detection.ipynb`에서 감지한 텍스트 영역을 크롭해 OCR을 실행합니다.

**처리 대상 타입:** `title` / `section_header` / `text` / `list_item` / `caption` / `page_header` / `page_footer`  
**건너뜀:** `formula` (→ 04), `table` (→ 05), `picture` (→ Model B)

In [ ]:
import fitz
import io
import os
import json
import re
import pytesseract
from PIL import Image, ImageFilter, ImageOps
import matplotlib.pyplot as plt

pytesseract.pytesseract.tesseract_cmd = r'C:\Program Files\Tesseract-OCR\tesseract.exe'

PDF_PATH   = "../data/sample/2015개정경제수학-광주교육청.pdf"
LAYOUT_DIR = "../outputs/layout"
OUTPUT_DIR = "../outputs/text_ocr"
os.makedirs(OUTPUT_DIR, exist_ok=True)

doc = fitz.open(PDF_PATH)
print(f"총 페이지 수: {len(doc)}")

In [ ]:
def page_to_image(doc, page_num, dpi=300):  # 크롭 OCR은 300 DPI 권장
    page = doc[page_num]
    pix = page.get_pixmap(dpi=dpi)
    return Image.open(io.BytesIO(pix.tobytes("png"))).convert("RGB")

def crop_element(img, bbox_px):
    x1, y1, x2, y2 = bbox_px
    x1, y1 = max(0, x1 - 6), max(0, y1 - 6)
    x2, y2 = min(img.width, x2 + 6), min(img.height, y2 + 6)
    return img.crop((x1, y1, x2, y2))

def preprocess(crop):
    """OCR 전 전처리: 흑백 + 샤프닝"""
    gray = ImageOps.grayscale(crop)
    sharp = gray.filter(ImageFilter.SHARPEN)
    return sharp

def clean_text(text):
    text = re.sub(r'([ \uAC00-\uD7A3A-Za-z0-9]) ([ \uAC00-\uD7A3A-Za-z0-9])', r'\1\2', text)
    text = re.sub(r'\n{3,}', '\n\n', text)
    return text.strip()

## 1. OCR 함수 정의

Tesseract는 이미 설치되어 있어 별도 모델 로드 없이 바로 실행됩니다.

In [ ]:
TEXT_TYPES = {"title", "sectionheader", "text", "listitem", "caption", "pageheader", "pagefooter", "handwriting"}
TESS_CONFIG = r'--oem 3 --psm 6'  # psm 6: 균일한 텍스트 블록 가정

def ocr_element(img, element):
    crop = crop_element(img, element["bbox_px"])
    processed = preprocess(crop)
    raw = pytesseract.image_to_string(processed, lang='kor+eng', config=TESS_CONFIG)
    return clean_text(raw)

## 2. 단일 페이지 테스트

In [ ]:
TEST_PAGE = 11  # 0-indexed (12페이지)

layout_path = os.path.join(LAYOUT_DIR, f"page_{TEST_PAGE+1}_layout.json")
with open(layout_path, encoding="utf-8") as f:
    page_data = json.load(f)

img = page_to_image(doc, TEST_PAGE)

text_elements = [e for e in page_data["elements"] if e["type"] in TEXT_TYPES]
print(f"텍스트 영역: {len(text_elements)}개 / 전체 {page_data['element_count']}개\n")

for elem in text_elements:
    text = ocr_element(img, elem)
    elem["text"] = text
    print(f"[{elem['id']}] {elem['type']:<16} → {repr(text[:60])}")

## 3. 크롭 + 추출 텍스트 시각화

In [ ]:
n = len(text_elements)
cols = 3
rows = (n + cols - 1) // cols

fig, axes = plt.subplots(rows, cols, figsize=(12, rows * 3))
axes = axes.flatten() if n > 1 else [axes]

for i, elem in enumerate(text_elements):
    crop = crop_element(img, elem["bbox_px"])
    axes[i].imshow(crop)
    axes[i].axis("off")
    label = f"[{elem['type']}]\n{repr(elem.get('text', '')[:40])}"
    axes[i].set_title(label, fontsize=7)

for j in range(n, len(axes)):
    axes[j].axis("off")

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, f"page_{TEST_PAGE+1}_crops.png"), dpi=120, bbox_inches="tight")
plt.show()

## 4. JSON 업데이트 저장

In [ ]:
out_path = os.path.join(OUTPUT_DIR, f"page_{TEST_PAGE+1}_text.json")
with open(out_path, "w", encoding="utf-8") as f:
    json.dump(page_data, f, ensure_ascii=False, indent=2)

print(f"저장 완료: {out_path}")
print(json.dumps(page_data, ensure_ascii=False, indent=2)[:1000])

## 5. 배치 처리

In [ ]:
batch_layout_path = os.path.join(LAYOUT_DIR, "batch_layout.json")
with open(batch_layout_path, encoding="utf-8") as f:
    all_pages = json.load(f)

for page_data in all_pages:
    page_num = page_data["page_id"] - 1
    print(f"[{page_num+1}페이지] 처리 중...", end=" ")

    img = page_to_image(doc, page_num)
    text_elems = [e for e in page_data["elements"] if e["type"] in TEXT_TYPES]

    for elem in text_elems:
        elem["text"] = ocr_element(img, elem)

    print(f"{len(text_elems)}개 OCR 완료")

batch_out = os.path.join(OUTPUT_DIR, "batch_text.json")
with open(batch_out, "w", encoding="utf-8") as f:
    json.dump(all_pages, f, ensure_ascii=False, indent=2)

print(f"\n배치 저장 완료: {batch_out}")